# 01 — Linear Regression From First Principles

This notebook implements Linear Regression from scratch using NumPy: train/test split, mean baseline, pseudo-inverse solution, gradient descent, Ridge Regression, and evaluation metrics.

In [ ]:
import numpy as np

## 1. Dataset

We create a synthetic regression dataset:

$$
y = \beta_0 + X\beta + \epsilon
$$

In [ ]:
rng = np.random.default_rng(42)

n = 180
X = rng.normal(0, 1, size=(n, 3))
true_beta = np.array([4.0, 2.5, -1.2, 0.8])
noise = rng.normal(0, 1.0, size=n)

y = true_beta[0] + X @ true_beta[1:] + noise

X.shape, y.shape

## 2. Train/Test Split

In [ ]:
def train_test_split_numpy(X, y, test_size=0.25, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y)
    indices = rng.permutation(n)
    test_n = int(n * test_size)
    test_idx = indices[:test_n]
    train_idx = indices[test_n:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X_train, X_test, y_train, y_test = train_test_split_numpy(X, y)

X_train.shape, X_test.shape

## 3. Metrics

In [ ]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


def rmse(y_true, y_pred):
    return np.sqrt(mse(y_true, y_pred))


def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot


def regression_report(y_true, y_pred):
    return mae(y_true, y_pred), mse(y_true, y_pred), rmse(y_true, y_pred), r2_score(y_true, y_pred)

## 4. Mean Baseline

A regression baseline can simply predict the training mean.

In [ ]:
baseline_pred = np.ones_like(y_test) * y_train.mean()
regression_report(y_test, baseline_pred)

## 5. Closed-Form Linear Regression

Using pseudo-inverse:

$$
\beta = X^+y
$$

In [ ]:
def add_bias_column(X):
    return np.column_stack([np.ones(X.shape[0]), X])


def linear_regression_pinv(X, y):
    X_bias = add_bias_column(X)
    beta = np.linalg.pinv(X_bias) @ y
    return beta


def predict_linear(X, beta):
    X_bias = add_bias_column(X)
    return X_bias @ beta

beta_pinv = linear_regression_pinv(X_train, y_train)
pred_pinv = predict_linear(X_test, beta_pinv)

beta_pinv, regression_report(y_test, pred_pinv)

## 6. Gradient Descent Linear Regression

$$
\beta \leftarrow \beta - \alpha \frac{2}{n}X^T(X\beta-y)
$$

In [ ]:
def linear_regression_gradient_descent(X, y, lr=0.05, steps=3000):
    X_bias = add_bias_column(X)
    beta = np.zeros(X_bias.shape[1])
    losses = []

    for step in range(steps):
        y_pred = X_bias @ beta
        error = y_pred - y
        losses.append(np.mean(error ** 2))
        gradient = (2 / len(y)) * X_bias.T @ error
        beta = beta - lr * gradient

    return beta, np.array(losses)

train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)
X_train_scaled = (X_train - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std

beta_gd, losses = linear_regression_gradient_descent(X_train_scaled, y_train)
pred_gd = predict_linear(X_test_scaled, beta_gd)

beta_gd, losses[0], losses[-1], regression_report(y_test, pred_gd)

## 7. Ridge Regression

$$
\beta_{ridge}=(X^TX+\lambda I)^{-1}X^Ty
$$

In [ ]:
def ridge_regression_closed_form(X, y, lambda_=1.0):
    X_bias = add_bias_column(X)
    identity = np.eye(X_bias.shape[1])
    identity[0, 0] = 0
    beta = np.linalg.solve(X_bias.T @ X_bias + lambda_ * identity, X_bias.T @ y)
    return beta

beta_ridge = ridge_regression_closed_form(X_train_scaled, y_train, lambda_=3.0)
pred_ridge = predict_linear(X_test_scaled, beta_ridge)

beta_ridge, regression_report(y_test, pred_ridge)

## 8. Residual Diagnostics

In [ ]:
residuals = y_test - pred_pinv

residuals.mean(), residuals.std(), np.max(np.abs(residuals))

## Reflection

Linear Regression is simple on the surface, but it connects loss functions, optimization, matrix algebra, probability, diagnostics, and generalization.